In [ ]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(784, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)
        x = self.fc(x)

        return x

In [ ]:
model = FashionCNN()

images = torch.randn(32, 1, 28, 28)

output = model(images)

print("Input shape: ", images.shape)
print("Output shape: ", output.shape)

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
print("Train size: ", len(train_dataset))
print("Test size: ", len(test_dataset))

image, label = train_dataset[0]

print("Image shape: ", image.shape)
print("Label: ", label)

In [ ]:
print(train_dataset.classes[9])

In [ ]:
train_subset, val_subset = random_split(
    train_dataset,
    [54000, 6000],
    generator=torch.Generator().manual_seed(42)
)

print("Training: ", len(train_subset))
print("Validation: ", len(val_subset))
print("Test: ", len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False
)

images, labels = next(iter(train_loader))

print("Images shape: ", images.shape)
print("Labels shape: ", labels.shape)
print("Train batches: ", len(train_loader))
print("Validation batches: ", len(val_loader))

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(9,5))

for i in range(10):
  plt.subplot(2, 5, i + 1)
  plt.imshow(images[i].squeeze(), cmap='gray')
  plt.title(train_dataset.classes[labels[i]])
  plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FashionCNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Device: ", device)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

model.train()

optimizer.zero_grad()

logits = model(images)

loss = loss_fn(logits, labels)

loss.backward()

optimizer.step()

print(f"Loss: {loss.item():.2f}")

In [ ]:
torch.manual_seed(42)

model = FashionCNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

model.train()

total_loss = 0
correct = 0
total = 0

for images, labels in train_loader:
  images = images.to(device)
  labels = labels.to(device)

  optimizer.zero_grad()
  logits = model(images)
  loss = loss_fn(logits, labels)
  loss.backward()
  optimizer.step()

  total_loss += loss.item() * images.size(0)
  predictions = torch.argmax(logits, dim=1)
  correct += (predictions == labels).sum().item()
  total += labels.size(0)

train_loss = total_loss / total
train_accuracy = correct / total

print(f"Train loss: {train_loss:.4f}")
print(f"Train accuracy: {train_accuracy:.2%}")

In [ ]:
model.eval()

val_total_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
  for images, labels in val_loader:
    images = images.to(device)
    labels = labels.to(device)

    logits = model(images)
    loss = loss_fn(logits, labels)

    val_total_loss += loss.item() * images.size(0)
    predictions = torch.argmax(logits, dim=1)
    val_correct += (predictions == labels).sum().item()
    val_total += labels.size(0)


val_loss = val_total_loss / val_total
val_accuracy = val_correct / val_total

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2%}")

In [ ]:
def evaluate_model(model, loader, loss_fn, device):
  model.eval()

  total_loss = 0
  correct = 0
  total = 0

  with torch.no_grad():
    for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      logits = model(images)
      loss = loss_fn(logits, labels)

      total_loss += loss.item() * images.size(0)
      predictions = torch.argmax(logits, dim=1)
      correct += (predictions == labels).sum().item()
      total += labels.size(0)

  loss = total_loss / total
  accuracy = correct / total

  return loss, accuracy

In [ ]:
val_loss, val_accuracy = evaluate_model(model, val_loader, loss_fn, device)

print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2%}")

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    device,
    epochs=5
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_accuracy = correct / total

        val_loss, val_accuracy = evaluate_model(model, val_loader, loss_fn, device)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)

        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Training Accuracy: {train_accuracy:.2%} | "
            f"Validation Accuracy: {val_accuracy:.2%}"
        )

    return history

In [ ]:
torch.manual_seed(42)

model = FashionCNN().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    epochs=5
)

In [ ]:
epochs_range = range(1, len(history["train_accuracy"]) + 1)

plt.figure(figsize=(11, 8))

plt.plot(
    epochs_range,
    history["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    epochs_range,
    history["val_accuracy"],
    marker="o",
    label="Validation Accuracy"
)

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Fashion-MNIST CNN Accuracy")


plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
epochs_range = range(1, len(history["train_accuracy"]) + 1)

plt.figure(figsize=(9, 5))

plt.plot(
    epochs_range,
    history["train_loss"],
    marker="o",
    label="Training Loss"
)

plt.plot(
    epochs_range,
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Fashion-MNIST CNN Loss")

plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

print("Test batches:", len(test_loader))

In [ ]:
test_loss, test_accuracy = evaluate_model(
    model,
    test_loader,
    loss_fn,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2%}")

In [ ]:
all_true = []
all_pred = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        predictions = torch.argmax(logits, dim=1)
        all_true.extend(labels.tolist())
        all_pred.extend(predictions.cpu().tolist())

print("True labels count:", len(all_true))
print("Predictions count:", len(all_pred))

In [ ]:
class_names = test_dataset.classes

cm = confusion_matrix(
    y_true=all_true,
    y_pred=all_pred,
    labels=list(range(10))
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(10, 9))

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=45
)

plt.title("Fashion-MNIST CNN Confusion Matrix")
plt.show()

In [ ]:

report = classification_report(
    all_true,
    all_pred,
    labels=list(range(10)),
    target_names=test_dataset.classes,
    digits=3
)

print(report)

In [ ]:
accuracy = sum(
    true == pred
    for true, pred in zip(all_true, all_pred)
) / len(all_true)

print(f"Current Accuracy: {accuracy:.2%}")

In [ ]:
wrong_indices = []

for i in range(len(all_true)):
    if all_true[i] != all_pred[i]:
        wrong_indices.append(i)

print("Wrong predictions:", len(wrong_indices))

In [ ]:
plt.figure(figsize=(12, 7))

for i in range(6):
    idx = wrong_indices[i]

    image, true_label = test_dataset[idx]
    predicted_label = all_pred[idx]

    plt.subplot(2, 3, i + 1)

    plt.imshow(image.squeeze(0), cmap="gray")

    plt.title(
        f"True: {class_names[true_label]}\n"
        f"Pred: {class_names[predicted_label]}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "fashion_cnn.pth")

In [ ]:
import os

print("Model file exists:", os.path.exists("fashion_cnn.pth"))

In [ ]:
saved_model = FashionCNN()

weights = torch.load(
    "fashion_cnn.pth",
    map_location="cpu",
    weights_only=True
)

saved_model.load_state_dict(weights)

saved_model.eval()

In [ ]:
saved_model.eval()

dummy_input = torch.randn(1, 1, 28, 28)

onnx_program = torch.onnx.export(
    saved_model,
    (dummy_input,),
    input_names=["images"],
    output_names=["logits"],
    opset_version=18,
    dynamo=True
)

onnx_program.save("fashion_cnn.onnx")

print("ONNX file exists:", os.path.exists("fashion_cnn.onnx"))

In [ ]:
import numpy as np
import onnxruntime as ort

image, label = test_dataset[0]
image = image.unsqueeze(0)

saved_model.eval()

with torch.no_grad():
    pytorch_output = saved_model(image).numpy()

session = ort.InferenceSession(
    "fashion_cnn.onnx",
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name

onnx_output = session.run(
    None,
    {input_name: image.numpy()}
)[0]

pytorch_pred = np.argmax(pytorch_output, axis=1)[0]
onnx_pred = np.argmax(onnx_output, axis=1)[0]

difference = np.max(np.abs(pytorch_output - onnx_output))

print("True class:", test_dataset.classes[label])
print("PyTorch prediction:", test_dataset.classes[pytorch_pred])
print("ONNX prediction:", test_dataset.classes[onnx_pred])
print("Maximum difference:", difference)

In [ ]:
from google.colab import files
from torchvision.transforms.functional import to_pil_image

sample_image, sample_label = test_dataset[0]

to_pil_image(sample_image).save("sample_ankle_boot.png")

print("Sample class:", test_dataset.classes[sample_label])

files.download("fashion_cnn.onnx")
files.download("sample_ankle_boot.png")